# 06 Recommendation Strategy

This notebook is the sixth step of the **AI-Powered Lead Generation & Recommendation Analytics Platform** project.

The goal of this stage is to design and compare recommendation strategies in a lead generation and live-commerce business context.

In previous notebooks, I created:

1. A cleaned order-level analytical dataset;
2. A simulated user interaction event log;
3. LLM-inspired intent classification results;
4. A model-based lead scoring dataset.

In this notebook, I use these outputs to design three recommendation strategies:

### Strategy A: Popularity-Based Recommendation

Recommend products that are popular across the whole marketplace.

### Strategy B: Category Preference Recommendation

Recommend products based on a user's historical category preference.

### Strategy C: Intent-Aware Recommendation

Recommend products by combining user intent, model-based lead score, category preference, seller quality, review score, and product performance.

The final recommendation output will be used in the next notebook for A/B test simulation and evaluation.

## Step 1: Import Libraries and Define Project Paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set random seed for reproducibility
np.random.seed(42)

# Define project paths
BASE_DIR = Path("..")

PROCESSED_DIR = BASE_DIR / "data" / "processed"
FINAL_DIR = BASE_DIR / "data" / "final"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLE_OUTPUT_DIR = OUTPUT_DIR / "tables"

# Create folders if they do not exist
FINAL_DIR.mkdir(parents=True, exist_ok=True)
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported and project paths defined successfully.")

Libraries imported and project paths defined successfully.


## Step 2: Load Input Datasets

In [2]:
# Load input datasets
order_base = pd.read_csv(PROCESSED_DIR / "clean_order_base.csv")
fact_user_events = pd.read_csv(FINAL_DIR / "fact_user_events.csv")
fact_lead_scores = pd.read_csv(FINAL_DIR / "fact_lead_scores.csv")

# Convert datetime columns
order_base["order_time"] = pd.to_datetime(order_base["order_time"])
fact_user_events["event_time"] = pd.to_datetime(fact_user_events["event_time"])
fact_lead_scores["order_time"] = pd.to_datetime(fact_lead_scores["order_time"])

print("Datasets loaded successfully.")
print("order_base shape:", order_base.shape)
print("fact_user_events shape:", fact_user_events.shape)
print("fact_lead_scores shape:", fact_lead_scores.shape)

Datasets loaded successfully.
order_base shape: (112650, 32)
fact_user_events shape: (386012, 15)
fact_lead_scores shape: (112650, 30)


## Step 3: Create Product Performance Table

Before designing recommendation strategies, I first create a product-level performance table.

Product performance signals include:

- Total orders;
- Unique buyers;
- Total GMV;
- Average GMV;
- Average review score;
- High-intent lead rate;
- Average model lead score.

This table helps identify products that are commercially strong and suitable for recommendation.

In [3]:
# Create product-level order performance
product_order_perf = (
    order_base[order_base["order_status"] == "delivered"]
    .groupby(["product_id", "category"])
    .agg(
        product_total_orders=("order_id", "nunique"),
        product_unique_buyers=("user_id", "nunique"),
        product_total_gmv=("gmv", "sum"),
        product_avg_gmv=("gmv", "mean"),
        product_avg_review_score=("review_score", "mean"),
        product_avg_price=("price", "mean")
    )
    .reset_index()
)

# Create product-level lead quality signals
product_lead_perf = (
    fact_lead_scores
    .groupby("product_id")
    .agg(
        product_avg_model_lead_score=("model_lead_score", "mean"),
        product_high_intent_rate=("high_intent_flag", "mean")
    )
    .reset_index()
)

# Merge product performance tables
product_performance = product_order_perf.merge(
    product_lead_perf,
    on="product_id",
    how="left"
)

# Fill missing lead signals
product_performance["product_avg_model_lead_score"] = product_performance["product_avg_model_lead_score"].fillna(
    product_performance["product_avg_model_lead_score"].median()
)

product_performance["product_high_intent_rate"] = product_performance["product_high_intent_rate"].fillna(0)

# Round numeric fields
round_cols = [
    "product_total_gmv",
    "product_avg_gmv",
    "product_avg_review_score",
    "product_avg_price",
    "product_avg_model_lead_score",
    "product_high_intent_rate"
]

for col in round_cols:
    product_performance[col] = product_performance[col].round(4)

product_performance.head()

,product_id,category,product_total_orders,product_unique_buyers,product_total_gmv,product_avg_gmv,product_avg_review_score,product_avg_price,product_avg_model_lead_score,product_high_intent_rate
0,00066f42aeeb9f3007548bb9d3f33c38,perfumery,1,1,120.24,120.24,5.0,101.65,99.750,1.0
1,00088930e925c41fd95ebfe695fd2655,auto,1,1,143.83,143.83,4.0,129.90,99.930,1.0
2,0009406fd7479715e4bef61dd91f2462,bed_bath_table,1,1,242.10,242.10,1.0,229.00,1.860,0.0
3,000b8f95fcb9e0096488278317764d19,housewares,2,2,157.00,78.50,5.0,58.90,99.235,1.0
4,000d9be29b5207b54e86aa1b1ac54872,watches_gifts,1,1,218.27,218.27,5.0,199.00,99.720,1.0


## Step 4: Create Seller Quality Table

Seller quality is important in lead generation recommendation.

A good recommendation should not only match users with relevant products, but also avoid pushing users toward sellers with poor fulfilment or low satisfaction.

Seller quality signals include:

- Seller total orders;
- Seller total GMV;
- Average review score;
- Late delivery rate;
- High-intent lead rate.

In [4]:
# Create seller-level order performance
seller_order_perf = (
    order_base[order_base["order_status"] == "delivered"]
    .groupby("seller_id")
    .agg(
        seller_total_orders=("order_id", "nunique"),
        seller_total_gmv=("gmv", "sum"),
        seller_avg_review_score=("review_score", "mean"),
        seller_late_delivery_rate=("late_delivery_flag", "mean")
    )
    .reset_index()
)

# Create seller-level lead quality
seller_lead_perf = (
    fact_lead_scores
    .groupby("seller_id")
    .agg(
        seller_avg_model_lead_score=("model_lead_score", "mean"),
        seller_high_intent_rate=("high_intent_flag", "mean")
    )
    .reset_index()
)

# Merge seller performance tables
seller_quality = seller_order_perf.merge(
    seller_lead_perf,
    on="seller_id",
    how="left"
)

# Fill missing values
seller_quality["seller_avg_model_lead_score"] = seller_quality["seller_avg_model_lead_score"].fillna(
    seller_quality["seller_avg_model_lead_score"].median()
)

seller_quality["seller_high_intent_rate"] = seller_quality["seller_high_intent_rate"].fillna(0)
seller_quality["seller_late_delivery_rate"] = seller_quality["seller_late_delivery_rate"].fillna(0)

# Round numeric fields
seller_quality["seller_total_gmv"] = seller_quality["seller_total_gmv"].round(2)
seller_quality["seller_avg_review_score"] = seller_quality["seller_avg_review_score"].round(4)
seller_quality["seller_late_delivery_rate"] = seller_quality["seller_late_delivery_rate"].round(4)
seller_quality["seller_avg_model_lead_score"] = seller_quality["seller_avg_model_lead_score"].round(4)
seller_quality["seller_high_intent_rate"] = seller_quality["seller_high_intent_rate"].round(4)

seller_quality.head()

,seller_id,seller_total_orders,seller_total_gmv,seller_avg_review_score,seller_late_delivery_rate,seller_avg_model_lead_score,seller_high_intent_rate
0,0015a82c2db000af6aaaf3ae2ecb0532,3,2748.06,3.6667,0.0000,97.2567,1.0000
1,001cca7ae9ae17fb1caed9dfb1094831,195,33142.90,3.9786,0.0513,88.9073,0.8954
2,002100f778ceb8431b7a1020ff7ab48f,50,1995.16,4.0370,0.1667,83.0078,0.8364
3,003554e2dce176b5555353e4f3555ac8,1,139.38,5.0000,0.0000,99.9300,1.0000
4,004c9cd9d87a3c30c522c48c4fc07416,156,23094.02,4.1667,0.0595,89.9244,0.9118


## Step 5: Create User Preference Table

A category preference recommendation strategy needs to understand each user's historical product category interests.

I infer user category preference from past purchases and interactions.

For each user-category pair, I calculate:

- Number of events;
- Number of purchases;
- Total GMV;
- Average lead score;
- High-intent rate.

The top category for each user will be used as their preferred category.

In [5]:
# Create user-category preference from lead score dataset
user_category_preference = (
    fact_lead_scores
    .groupby(["user_id", "category"])
    .agg(
        user_category_records=("order_id", "count"),
        user_category_total_gmv=("gmv", "sum"),
        user_category_avg_lead_score=("model_lead_score", "mean"),
        user_category_high_intent_rate=("high_intent_flag", "mean")
    )
    .reset_index()
)

# Sort to find each user's preferred category
user_category_preference = user_category_preference.sort_values(
    by=[
        "user_id",
        "user_category_records",
        "user_category_total_gmv",
        "user_category_avg_lead_score"
    ],
    ascending=[True, False, False, False]
)

# Keep top category per user
user_top_category = (
    user_category_preference
    .groupby("user_id")
    .head(1)
    .reset_index(drop=True)
    .rename(columns={"category": "preferred_category"})
)

user_top_category.head()

,user_id,preferred_category,user_category_records,user_category_total_gmv,user_category_avg_lead_score,user_category_high_intent_rate
0,0000366f3b9a7992bf8c76cfdf3221e2,bed_bath_table,1,141.90,99.19,1.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,health_beauty,1,27.19,99.99,1.0
2,0000f46a3911fa3c0805444483337064,stationery,1,86.22,0.68,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,telephony,1,43.62,99.99,1.0
4,0004aac84e0df4da2b147fca70cf8255,telephony,1,196.89,99.78,1.0


## Step 6: Create Candidate Product Pool

Recommendation systems usually generate candidate items first, then rank them.

In this project, I create a candidate product pool from products with enough historical performance data.

To avoid recommending extremely sparse products, I keep products with at least two historical orders.

The candidate pool will be used by all three recommendation strategies.

In [6]:
# Create candidate product pool
candidate_products = product_performance[
    product_performance["product_total_orders"] >= 2
].copy()

# Merge seller quality by finding the main seller for each product
product_main_seller = (
    order_base[order_base["order_status"] == "delivered"]
    .groupby(["product_id", "seller_id"])
    .agg(product_seller_orders=("order_id", "nunique"))
    .reset_index()
    .sort_values(["product_id", "product_seller_orders"], ascending=[True, False])
    .groupby("product_id")
    .head(1)
    .reset_index(drop=True)
)

candidate_products = candidate_products.merge(
    product_main_seller[["product_id", "seller_id"]],
    on="product_id",
    how="left"
)

candidate_products = candidate_products.merge(
    seller_quality,
    on="seller_id",
    how="left"
)

# Fill seller missing values
candidate_products["seller_avg_review_score"] = candidate_products["seller_avg_review_score"].fillna(
    candidate_products["seller_avg_review_score"].median()
)
candidate_products["seller_late_delivery_rate"] = candidate_products["seller_late_delivery_rate"].fillna(0)
candidate_products["seller_high_intent_rate"] = candidate_products["seller_high_intent_rate"].fillna(0)

print("Candidate product pool shape:", candidate_products.shape)
candidate_products.head()

Candidate product pool shape: (13101, 17)


,product_id,category,product_total_orders,product_unique_buyers,product_total_gmv,product_avg_gmv,product_avg_review_score,product_avg_price,product_avg_model_lead_score,product_high_intent_rate,seller_id,seller_total_orders,seller_total_gmv,seller_avg_review_score,seller_late_delivery_rate,seller_avg_model_lead_score,seller_high_intent_rate
0,000b8f95fcb9e0096488278317764d19,housewares,2,2,157.00,78.5000,5.0000,58.9000,99.2350,1.0000,40ec8ab6cdafbcc4f544da38c67da39a,2,157.00,5.0000,0.0000,99.2350,1.0000
1,00126f27c813603687e6ce486d909d01,cool_stuff,2,2,527.73,263.8650,5.0000,249.0000,99.8900,1.0000,cd68562d3f44870c08922d380acae552,122,20279.86,3.9147,0.1783,82.4763,0.8333
2,001795ec6f1b187d37335e1c4704762e,consoles_games,7,7,456.20,50.6889,3.2222,38.9000,87.8122,0.8889,8b321bb669392f5163d04c59e235e066,930,31186.94,4.0259,0.0866,88.2225,0.8900
3,001b72dfd63e9833e8c02742adf472e3,furniture_decor,11,11,620.83,47.7562,3.6923,34.9900,77.4157,0.7857,8a32e327fe2c1b3511609d81aaf9f042,126,8338.86,3.9527,0.0592,84.0728,0.8500
4,00210e41887c2a8ef9f791ebc780cc36,health_beauty,5,5,319.10,45.5857,4.4286,33.4129,85.6871,0.8571,e9779976487b77c6d4ac45f75ec7afe9,645,52211.30,4.2073,0.0623,91.2421,0.9200


## Step 7: Define Normalisation Helper Function

To combine different ranking signals into one recommendation score, I normalise numeric columns to a 0-1 range.

This allows signals such as GMV, order count, review score, lead score, and seller quality to be combined in a comparable way.

In [7]:
def min_max_normalize(series):
    """
    Apply min-max normalization to a numeric Pandas Series.
    
    If all values are the same, return 0.5 for all records to avoid division by zero.
    """
    min_value = series.min()
    max_value = series.max()
    
    if max_value == min_value:
        return pd.Series([0.5] * len(series), index=series.index)
    
    return (series - min_value) / (max_value - min_value)

## Step 8: Build Popularity-Based Recommendation Score

Strategy A recommends products based on overall popularity.

This is the simplest recommendation baseline.

Signals used:

- Product total orders;
- Product total GMV;
- Product average review score.

This strategy does not consider individual user preference or intent.

In [8]:
# Create normalized popularity signals
candidate_products["norm_product_orders"] = min_max_normalize(candidate_products["product_total_orders"])
candidate_products["norm_product_gmv"] = min_max_normalize(candidate_products["product_total_gmv"])
candidate_products["norm_product_review"] = min_max_normalize(candidate_products["product_avg_review_score"])

# Popularity score
candidate_products["popularity_score"] = (
    0.45 * candidate_products["norm_product_orders"] +
    0.35 * candidate_products["norm_product_gmv"] +
    0.20 * candidate_products["norm_product_review"]
)

# Sort global popular products
popular_products_ranked = candidate_products.sort_values(
    by="popularity_score",
    ascending=False
).reset_index(drop=True)

popular_products_ranked[[
    "product_id",
    "category",
    "seller_id",
    "product_total_orders",
    "product_total_gmv",
    "product_avg_review_score",
    "popularity_score"
]].head(10)

,product_id,category,seller_id,product_total_orders,product_total_gmv,product_avg_review_score,popularity_score
0,99a4788cb24856965c36a24e339b6058,bed_bath_table,4a3ca9315b744ce9f8e9374361493884,456,49907.50,3.9444,0.856891
1,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,955fee9216a65b617aa5c0531780ce60,425,44197.75,4.0538,0.801909
2,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,a1043bafd471dff536d0c462352beb48,313,58957.31,4.2892,0.779506
3,bb50f2e236e5eea0100680137654686c,health_beauty,f7ba60f8c3f99e7ee4042fdef03b70c4,186,67258.03,4.2371,0.694234
4,422879e10f46682990de24d770e7f83d,garden_tools,1f50f920176fa81dab994f9023523100,352,34201.26,3.9649,0.673065
5,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,7d13fca15225358621be4086e1eb0964,304,39713.49,4.2056,0.666220
6,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,de722cd6dad950a92b7d4f82673f8833,253,47876.06,4.2243,0.659099
7,6cdd53843498f92890544667809f1595,health_beauty,ccc4bbb5f32a6ab2b7066a4130f114e3,148,57933.73,4.3660,0.614470
8,389d119b48cf3043d311335e499d9c6b,garden_tools,1f50f920176fa81dab994f9023523100,309,28543.65,4.1359,0.609539
9,368c6c730842d78016ad823897a372db,garden_tools,1f50f920176fa81dab994f9023523100,291,27984.40,3.9317,0.578576


## Step 9: Build Intent-Aware Recommendation Score

Strategy C uses both product quality and lead generation signals.

Signals used:

- Product popularity;
- Product review score;
- Product high-intent rate;
- Average model lead score;
- Seller review score;
- Seller high-intent rate;
- Seller late delivery rate penalty.

This strategy is closer to a lead generation recommendation context because it prioritises products and sellers that are more likely to convert high-intent users.

In [9]:
# Normalize intent-aware signals
candidate_products["norm_product_high_intent_rate"] = min_max_normalize(candidate_products["product_high_intent_rate"])
candidate_products["norm_product_lead_score"] = min_max_normalize(candidate_products["product_avg_model_lead_score"])
candidate_products["norm_seller_review"] = min_max_normalize(candidate_products["seller_avg_review_score"])
candidate_products["norm_seller_high_intent_rate"] = min_max_normalize(candidate_products["seller_high_intent_rate"])
candidate_products["norm_seller_late_delivery_penalty"] = 1 - min_max_normalize(candidate_products["seller_late_delivery_rate"])

# Intent-aware base score
candidate_products["intent_aware_score"] = (
    0.20 * candidate_products["norm_product_orders"] +
    0.15 * candidate_products["norm_product_gmv"] +
    0.15 * candidate_products["norm_product_review"] +
    0.20 * candidate_products["norm_product_high_intent_rate"] +
    0.15 * candidate_products["norm_product_lead_score"] +
    0.10 * candidate_products["norm_seller_review"] +
    0.10 * candidate_products["norm_seller_high_intent_rate"] +
    0.05 * candidate_products["norm_seller_late_delivery_penalty"]
)

intent_products_ranked = candidate_products.sort_values(
    by="intent_aware_score",
    ascending=False
).reset_index(drop=True)

intent_products_ranked[[
    "product_id",
    "category",
    "seller_id",
    "product_total_orders",
    "product_total_gmv",
    "product_avg_review_score",
    "product_high_intent_rate",
    "product_avg_model_lead_score",
    "seller_avg_review_score",
    "seller_high_intent_rate",
    "seller_late_delivery_rate",
    "intent_aware_score"
]].head(10)

,product_id,category,seller_id,product_total_orders,product_total_gmv,product_avg_review_score,product_high_intent_rate,product_avg_model_lead_score,seller_avg_review_score,seller_high_intent_rate,seller_late_delivery_rate,intent_aware_score
0,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,955fee9216a65b617aa5c0531780ce60,425,44197.75,4.0538,0.9108,90.4134,4.0944,0.9113,0.0645,0.932423
1,99a4788cb24856965c36a24e339b6058,bed_bath_table,4a3ca9315b744ce9f8e9374361493884,456,49907.50,3.9444,0.8770,86.7273,3.8415,0.8757,0.0970,0.930907
2,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,a1043bafd471dff536d0c462352beb48,313,58957.31,4.2892,0.9067,89.4698,4.2487,0.9117,0.0492,0.927266
3,422879e10f46682990de24d770e7f83d,garden_tools,1f50f920176fa81dab994f9023523100,352,34201.26,3.9649,0.9463,93.8325,3.9966,0.9244,0.0779,0.885061
4,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,de722cd6dad950a92b7d4f82673f8833,253,47876.06,4.2243,0.9307,91.8065,4.1698,0.9472,0.0318,0.884437
5,bb50f2e236e5eea0100680137654686c,health_beauty,f7ba60f8c3f99e7ee4042fdef03b70c4,186,67258.03,4.2371,0.9026,89.1020,4.2183,0.9000,0.1310,0.880493
6,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,7d13fca15225358621be4086e1eb0964,304,39713.49,4.2056,0.9257,91.9987,4.0228,0.9135,0.1121,0.876219
7,6cdd53843498f92890544667809f1595,health_beauty,ccc4bbb5f32a6ab2b7066a4130f114e3,148,57933.73,4.3660,0.9167,90.5919,4.3228,0.9115,0.0635,0.859980
8,389d119b48cf3043d311335e499d9c6b,garden_tools,1f50f920176fa81dab994f9023523100,309,28543.65,4.1359,0.9158,90.9735,3.9966,0.9244,0.0779,0.849510
9,368c6c730842d78016ad823897a372db,garden_tools,1f50f920176fa81dab994f9023523100,291,27984.40,3.9317,0.9278,92.0423,3.9966,0.9244,0.0779,0.836682


## Step 10: Prepare Target Users for Recommendation

I create a user-level table for users who will receive recommendations.

User-level fields include:

- Preferred category;
- User value segment;
- Average model lead score;
- Historical high-intent rate;
- Total GMV.

These fields help support user-level recommendation generation and later A/B test simulation.

## Computational Efficiency Note

The full dataset contains a large number of users. To keep the notebook efficient and reproducible on a local machine, this project generates recommendations for a representative sample of users.

This does not change the recommendation strategy logic. In a production environment, the same logic can be scaled through batch processing or distributed computation.

In [15]:
# Create user-level recommendation profile
user_profile = (
    fact_lead_scores
    .groupby("user_id")
    .agg(
        user_total_records=("order_id", "count"),
        user_total_gmv=("gmv", "sum"),
        user_avg_model_lead_score=("model_lead_score", "mean"),
        user_high_intent_rate=("high_intent_flag", "mean"),
        user_value_segment=("user_value_segment", "first")
    )
    .reset_index()
)

user_profile = user_profile.merge(
    user_top_category[["user_id", "preferred_category"]],
    on="user_id",
    how="left"
)

user_profile["preferred_category"] = user_profile["preferred_category"].fillna("unknown")

# Use a representative sample of users to keep recommendation generation efficient
sample_size = min(5000, len(user_profile))

target_users = (
    user_profile
    .sample(n=sample_size, random_state=42)
    .reset_index(drop=True)
)

print("Total users:", user_profile.shape[0])
print("Target users for recommendation generation:", target_users.shape[0])

target_users.head()

Total users: 95420
Target users for recommendation generation: 5000


,user_id,user_total_records,user_total_gmv,user_avg_model_lead_score,user_high_intent_rate,user_value_segment,preferred_category
0,842bff27a9fd73c774aeb526d0f53113,1,63.70,99.55,1.0,Low Value,bed_bath_table
1,97e5be9a2f1841bc42b966aa2d014d13,1,118.72,99.93,1.0,Medium Value,industry_commerce_and_business
2,987f4716cef1294c01c4c0cb7b275ff1,1,200.17,99.83,1.0,High Value,bed_bath_table
3,10e1df2717597281c5cffa0e0a7ca7cb,1,2225.69,99.50,1.0,High Value,bed_bath_table
4,6baee3008124f2fd07c61fc8169ad485,3,285.63,100.00,1.0,High Value,sports_leisure


## Step 11: Generate Recommendations

For each user, I generate one recommendation from each strategy:

1. Popularity-based recommendation;
2. Category preference recommendation;
3. Intent-aware recommendation.

To avoid recommending products the user has already purchased before, I exclude products that appear in the user's historical records.

In [19]:
# This version pre-computes top candidate pools to avoid scanning the full product table for every user.

# Pre-compute top global popular products
top_popular_products = popular_products_ranked.head(1000).copy()

# Pre-compute top intent-aware products
top_intent_products = intent_products_ranked.head(1000).copy()

# Pre-compute top products by category for category-preference recommendation
top_category_products = {}

for category, group in candidate_products.groupby("category"):
    top_category_products[category] = (
        group
        .sort_values(by="popularity_score", ascending=False)
        .head(200)
        .copy()
    )

# Create user historical purchased products
user_purchased_products = (
    fact_lead_scores
    .groupby("user_id")["product_id"]
    .apply(set)
    .to_dict()
)

recommendation_records = []

for _, user in target_users.iterrows():
    user_id = user["user_id"]
    preferred_category = user["preferred_category"]
    user_segment = user["user_value_segment"]
    user_avg_lead_score = user["user_avg_model_lead_score"]
    user_high_intent_rate = user["user_high_intent_rate"]
    
    purchased_set = user_purchased_products.get(user_id, set())
    
 
    # Strategy A: Popularity-based
   
    popularity_candidates = top_popular_products[
        ~top_popular_products["product_id"].isin(purchased_set)
    ]
    
    if not popularity_candidates.empty:
        top_popular = popularity_candidates.iloc[0]
        
        recommendation_records.append({
            "user_id": user_id,
            "user_value_segment": user_segment,
            "preferred_category": preferred_category,
            "strategy": "popularity_based",
            "recommended_product_id": top_popular["product_id"],
            "recommended_seller_id": top_popular["seller_id"],
            "recommended_category": top_popular["category"],
            "recommendation_score": round(top_popular["popularity_score"], 4),
            "product_avg_price": round(top_popular["product_avg_price"], 2),
            "product_avg_review_score": round(top_popular["product_avg_review_score"], 4),
            "product_high_intent_rate": round(top_popular["product_high_intent_rate"], 4),
            "seller_avg_review_score": round(top_popular["seller_avg_review_score"], 4),
            "seller_late_delivery_rate": round(top_popular["seller_late_delivery_rate"], 4),
            "user_avg_model_lead_score": round(user_avg_lead_score, 4),
            "user_high_intent_rate": round(user_high_intent_rate, 4)
        })

    
    # Strategy B: Category Preference
  
    category_candidates = top_category_products.get(
        preferred_category,
        top_popular_products
    )
    
    category_candidates = category_candidates[
        ~category_candidates["product_id"].isin(purchased_set)
    ]
    
    # If no product exists in the preferred category, fall back to popularity-based candidates
    if category_candidates.empty:
        category_candidates = popularity_candidates
    
    if not category_candidates.empty:
        top_category = category_candidates.iloc[0]
        
        recommendation_records.append({
            "user_id": user_id,
            "user_value_segment": user_segment,
            "preferred_category": preferred_category,
            "strategy": "category_preference",
            "recommended_product_id": top_category["product_id"],
            "recommended_seller_id": top_category["seller_id"],
            "recommended_category": top_category["category"],
            "recommendation_score": round(top_category["popularity_score"], 4),
            "product_avg_price": round(top_category["product_avg_price"], 2),
            "product_avg_review_score": round(top_category["product_avg_review_score"], 4),
            "product_high_intent_rate": round(top_category["product_high_intent_rate"], 4),
            "seller_avg_review_score": round(top_category["seller_avg_review_score"], 4),
            "seller_late_delivery_rate": round(top_category["seller_late_delivery_rate"], 4),
            "user_avg_model_lead_score": round(user_avg_lead_score, 4),
            "user_high_intent_rate": round(user_high_intent_rate, 4)
        })
    
    
    # Strategy C: Intent-aware
    
    intent_candidates = top_intent_products[
        ~top_intent_products["product_id"].isin(purchased_set)
    ].copy()
    
    if not intent_candidates.empty:
        # Add category match bonus
        intent_candidates["category_match_bonus"] = np.where(
            intent_candidates["category"] == preferred_category,
            0.10,
            0.00
        )
        
        # Add user segment bonus
        if user_segment == "High Value":
            user_segment_bonus = 0.05
        elif user_segment == "Medium Value":
            user_segment_bonus = 0.03
        else:
            user_segment_bonus = 0.00
        
        intent_candidates["final_intent_recommendation_score"] = (
            intent_candidates["intent_aware_score"] +
            intent_candidates["category_match_bonus"] +
            user_segment_bonus
        )
        
        top_intent = intent_candidates.sort_values(
            by="final_intent_recommendation_score",
            ascending=False
        ).iloc[0]
        
        recommendation_records.append({
            "user_id": user_id,
            "user_value_segment": user_segment,
            "preferred_category": preferred_category,
            "strategy": "intent_aware",
            "recommended_product_id": top_intent["product_id"],
            "recommended_seller_id": top_intent["seller_id"],
            "recommended_category": top_intent["category"],
            "recommendation_score": round(top_intent["final_intent_recommendation_score"], 4),
            "product_avg_price": round(top_intent["product_avg_price"], 2),
            "product_avg_review_score": round(top_intent["product_avg_review_score"], 4),
            "product_high_intent_rate": round(top_intent["product_high_intent_rate"], 4),
            "seller_avg_review_score": round(top_intent["seller_avg_review_score"], 4),
            "seller_late_delivery_rate": round(top_intent["seller_late_delivery_rate"], 4),
            "user_avg_model_lead_score": round(user_avg_lead_score, 4),
            "user_high_intent_rate": round(user_high_intent_rate, 4)
        })

# Convert to DataFrame
fact_recommendations = pd.DataFrame(recommendation_records)

print("Recommendation records generated:", fact_recommendations.shape)
fact_recommendations.head(10)

Recommendation records generated: (15000, 15)


,user_id,user_value_segment,preferred_category,strategy,recommended_product_id,recommended_seller_id,recommended_category,recommendation_score,product_avg_price,product_avg_review_score,product_high_intent_rate,seller_avg_review_score,seller_late_delivery_rate,user_avg_model_lead_score,user_high_intent_rate
0,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,99.55,1.0
1,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,category_preference,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,99.55,1.0
2,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,intent_aware,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,1.0309,88.15,3.9444,0.8770,3.8415,0.0970,99.55,1.0
3,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,99.93,1.0
4,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,category_preference,3986c6e32f39114eff1ded07f1cb16fb,744dac408745240a2c2528fb1b6028f3,industry_commerce_and_business,0.2079,748.00,5.0000,1.0000,4.5542,0.0120,99.93,1.0
5,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,intent_aware,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,furniture_decor,0.9624,71.35,4.0538,0.9108,4.0944,0.0645,99.93,1.0
6,987f4716cef1294c01c4c0cb7b275ff1,High Value,bed_bath_table,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,99.83,1.0
7,987f4716cef1294c01c4c0cb7b275ff1,High Value,bed_bath_table,category_preference,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,99.83,1.0
8,987f4716cef1294c01c4c0cb7b275ff1,High Value,bed_bath_table,intent_aware,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,1.0809,88.15,3.9444,0.8770,3.8415,0.0970,99.83,1.0
9,10e1df2717597281c5cffa0e0a7ca7cb,High Value,bed_bath_table,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,99.50,1.0


## Step 12: Validate Recommendation Output

I check how many recommendations were generated by each strategy.

Since I generate one recommendation per strategy per user, the number of records should be approximately three times the number of target users.

In [20]:
# Validate recommendation output
recommendation_counts = (
    fact_recommendations
    .groupby("strategy")
    .agg(
        recommendations=("recommended_product_id", "count"),
        unique_users=("user_id", "nunique"),
        unique_products=("recommended_product_id", "nunique"),
        unique_sellers=("recommended_seller_id", "nunique"),
        avg_recommendation_score=("recommendation_score", "mean"),
        avg_product_review_score=("product_avg_review_score", "mean"),
        avg_product_high_intent_rate=("product_high_intent_rate", "mean"),
        avg_seller_review_score=("seller_avg_review_score", "mean"),
        avg_seller_late_delivery_rate=("seller_late_delivery_rate", "mean")
    )
    .reset_index()
)

# Round fields
for col in [
    "avg_recommendation_score",
    "avg_product_review_score",
    "avg_product_high_intent_rate",
    "avg_seller_review_score",
    "avg_seller_late_delivery_rate"
]:
    recommendation_counts[col] = recommendation_counts[col].round(4)

recommendation_counts

,strategy,recommendations,unique_users,unique_products,unique_sellers,avg_recommendation_score,avg_product_review_score,avg_product_high_intent_rate,avg_seller_review_score,avg_seller_late_delivery_rate
0,category_preference,5000,5000,105,83,0.4821,4.1977,0.9083,4.1444,0.0717
1,intent_aware,5000,5000,10,8,0.9871,4.0822,0.9086,4.0854,0.0749
2,popularity_based,5000,5000,2,2,0.8567,3.9448,0.8771,3.8425,0.0969


## Step 13: Compare Strategy-Level Recommendation Quality

This summary compares the quality of products and sellers selected by different strategies.

Useful comparison metrics:

- Average product review score;
- Product high-intent rate;
- Seller review score;
- Seller late delivery rate;
- Product diversity;
- Seller diversity.

In a lead generation context, a better recommendation strategy should balance popularity, relevance, user intent, seller quality, and product diversity.

## Strategy Comparison Insight

The popularity-based strategy generated highly concentrated recommendations, with only a small number of unique products receiving most exposure. This reflects a common limitation of popularity-based recommendation systems: they are simple and effective as a baseline, but may reduce product diversity and overlook individual user preference.

The category-preference strategy improved product diversity by recommending products from each user's preferred category.

The intent-aware strategy balanced product performance, seller quality, lead score, and category match. It is more aligned with lead generation objectives because it prioritises products and sellers that are more likely to attract high-intent users.

In [21]:
recommendation_strategy_summary = recommendation_counts.copy()

# Add diversity metrics
total_users = fact_recommendations["user_id"].nunique()

recommendation_strategy_summary["product_diversity_rate"] = (
    recommendation_strategy_summary["unique_products"] / total_users
).round(4)

recommendation_strategy_summary["seller_diversity_rate"] = (
    recommendation_strategy_summary["unique_sellers"] / total_users
).round(4)

recommendation_strategy_summary = recommendation_strategy_summary.sort_values(
    by="avg_product_high_intent_rate",
    ascending=False
)

recommendation_strategy_summary

,strategy,recommendations,unique_users,unique_products,unique_sellers,avg_recommendation_score,avg_product_review_score,avg_product_high_intent_rate,avg_seller_review_score,avg_seller_late_delivery_rate,product_diversity_rate,seller_diversity_rate
1,intent_aware,5000,5000,10,8,0.9871,4.0822,0.9086,4.0854,0.0749,0.0020,0.0016
0,category_preference,5000,5000,105,83,0.4821,4.1977,0.9083,4.1444,0.0717,0.0210,0.0166
2,popularity_based,5000,5000,2,2,0.8567,3.9448,0.8771,3.8425,0.0969,0.0004,0.0004


In [22]:
# Export recommendation strategy summary
recommendation_strategy_summary.to_csv(
    TABLE_OUTPUT_DIR / "recommendation_strategy_summary.csv",
    index=False
)

print("recommendation_strategy_summary.csv exported successfully.")

recommendation_strategy_summary.csv exported successfully.


## Step 14: Create User-Level Recommendation Summary

This output summarises recommendations at the user level.

It helps check whether each user received recommendations from all three strategies and whether recommendation categories align with user preferences.

In [23]:
# Create user-level recommendation summary
recommendation_user_level = (
    fact_recommendations
    .groupby(["user_id", "user_value_segment", "preferred_category"])
    .agg(
        strategies_received=("strategy", "nunique"),
        recommended_products=("recommended_product_id", "nunique"),
        recommended_categories=("recommended_category", "nunique"),
        avg_recommendation_score=("recommendation_score", "mean"),
        avg_product_high_intent_rate=("product_high_intent_rate", "mean"),
        avg_seller_review_score=("seller_avg_review_score", "mean")
    )
    .reset_index()
)

recommendation_user_level["avg_recommendation_score"] = recommendation_user_level["avg_recommendation_score"].round(4)
recommendation_user_level["avg_product_high_intent_rate"] = recommendation_user_level["avg_product_high_intent_rate"].round(4)
recommendation_user_level["avg_seller_review_score"] = recommendation_user_level["avg_seller_review_score"].round(4)

recommendation_user_level.head()

,user_id,user_value_segment,preferred_category,strategies_received,recommended_products,recommended_categories,avg_recommendation_score,avg_product_high_intent_rate,avg_seller_review_score
0,002feefec5af0a3b26ee7839c66d205e,Medium Value,garden_tools,3,2,2,0.8484,0.9232,3.9449
1,0037d2bbb9ebc39d9114aea27ee16d72,Medium Value,furniture_decor,3,2,2,0.9071,0.8995,4.0101
2,003d56767e53e08671de00da3fba8d40,High Value,bed_bath_table,3,1,1,0.9316,0.8770,3.8415
3,004b9b1634b7becda8eec4e96bced882,High Value,sports_leisure,3,3,3,0.7082,0.9015,4.0378
4,004ed862ea4e55f18868d3d782d10879,Medium Value,luggage_accessories,3,3,3,0.7117,0.9232,3.9884


In [24]:
# Export user-level recommendation summary
recommendation_user_level.to_csv(
    TABLE_OUTPUT_DIR / "recommendation_user_level.csv",
    index=False
)

print("recommendation_user_level.csv exported successfully.")

recommendation_user_level.csv exported successfully.


## Step 15: Create Recommendation Category Summary

This output summarises recommendation distribution by strategy and recommended category.

It helps evaluate whether a strategy over-concentrates recommendations in a few categories or provides broader category coverage.

In [25]:
# Create recommendation category summary
recommendation_category_summary = (
    fact_recommendations
    .groupby(["strategy", "recommended_category"])
    .agg(
        recommendations=("recommended_product_id", "count"),
        unique_users=("user_id", "nunique"),
        unique_products=("recommended_product_id", "nunique"),
        avg_recommendation_score=("recommendation_score", "mean"),
        avg_product_high_intent_rate=("product_high_intent_rate", "mean"),
        avg_product_review_score=("product_avg_review_score", "mean"),
        avg_seller_review_score=("seller_avg_review_score", "mean")
    )
    .reset_index()
)

# Round numeric fields
for col in [
    "avg_recommendation_score",
    "avg_product_high_intent_rate",
    "avg_product_review_score",
    "avg_seller_review_score"
]:
    recommendation_category_summary[col] = recommendation_category_summary[col].round(4)

recommendation_category_summary = recommendation_category_summary.sort_values(
    by=["strategy", "recommendations"],
    ascending=[True, False]
)

recommendation_category_summary.head(20)

,strategy,recommended_category,recommendations,unique_users,unique_products,avg_recommendation_score,avg_product_high_intent_rate,avg_product_review_score,avg_seller_review_score
7,category_preference,bed_bath_table,451,451,2,0.8421,0.8793,3.9620,3.8610
39,category_preference,health_beauty,423,423,2,0.6919,0.9030,4.2408,4.2213
58,category_preference,sports_leisure,380,380,2,0.2851,0.9168,4.1537,4.1731
14,category_preference,computers_accessories,342,342,2,0.7749,0.9076,4.2867,4.2457
44,category_preference,housewares,320,320,2,0.3705,0.9068,4.0145,4.0290
35,category_preference,furniture_decor,314,314,2,0.7642,0.9147,4.0795,4.1115
64,category_preference,watches_gifts,294,294,2,0.6599,0.9222,4.1895,4.0278
5,category_preference,auto,203,203,3,0.3279,0.9331,4.3307,4.2687
61,category_preference,telephony,200,200,2,0.2945,0.9805,4.3135,4.1325
62,category_preference,toys,197,197,2,0.2888,0.9185,4.3662,4.4428


In [26]:
# Export recommendation category summary
recommendation_category_summary.to_csv(
    TABLE_OUTPUT_DIR / "recommendation_category_summary.csv",
    index=False
)

print("recommendation_category_summary.csv exported successfully.")

recommendation_category_summary.csv exported successfully.


## Step 16: Add Recommendation Strategy Labels for A/B Test Preparation

The next notebook will simulate A/B testing.

To prepare for that, I map strategies to experiment groups:

- Control group: popularity-based recommendation;
- Treatment group 1: category preference recommendation;
- Treatment group 2: intent-aware recommendation.

The main A/B comparison will focus on:

- Control: popularity-based;
- Treatment: intent-aware.

In [27]:
# Map recommendation strategies to experiment group labels
strategy_group_map = {
    "popularity_based": "control",
    "category_preference": "treatment_category",
    "intent_aware": "treatment_intent"
}

fact_recommendations["experiment_group"] = fact_recommendations["strategy"].map(strategy_group_map)

# Add recommendation date for experiment simulation
fact_recommendations["recommendation_date"] = pd.Timestamp("2026-01-01")

fact_recommendations[[
    "user_id",
    "strategy",
    "experiment_group",
    "recommended_product_id",
    "recommended_category",
    "recommendation_score",
    "recommendation_date"
]].head()

,user_id,strategy,experiment_group,recommended_product_id,recommended_category,recommendation_score,recommendation_date
0,842bff27a9fd73c774aeb526d0f53113,popularity_based,control,99a4788cb24856965c36a24e339b6058,bed_bath_table,0.8569,2026-01-01
1,842bff27a9fd73c774aeb526d0f53113,category_preference,treatment_category,99a4788cb24856965c36a24e339b6058,bed_bath_table,0.8569,2026-01-01
2,842bff27a9fd73c774aeb526d0f53113,intent_aware,treatment_intent,99a4788cb24856965c36a24e339b6058,bed_bath_table,1.0309,2026-01-01
3,97e5be9a2f1841bc42b966aa2d014d13,popularity_based,control,99a4788cb24856965c36a24e339b6058,bed_bath_table,0.8569,2026-01-01
4,97e5be9a2f1841bc42b966aa2d014d13,category_preference,treatment_category,3986c6e32f39114eff1ded07f1cb16fb,industry_commerce_and_business,0.2079,2026-01-01


In [28]:
# Export final recommendation dataset
fact_recommendations.to_csv(
    FINAL_DIR / "fact_recommendations.csv",
    index=False
)

print("fact_recommendations.csv exported successfully.")
print("File saved to:", FINAL_DIR / "fact_recommendations.csv")

fact_recommendations.csv exported successfully.
File saved to: ../data/final/fact_recommendations.csv


## Step 17: Summary of Recommendation Strategy Design

In this notebook, I designed and compared three recommendation strategies using a representative sample of 5,000 users for efficient local computation.

The three strategies are:

1. **Popularity-Based Recommendation**
   - Recommends globally popular products.
   - Useful as a simple baseline.
   - Does not consider user preference or lead intent.
   - In this project, it generated highly concentrated exposure on a small number of products.

2. **Category Preference Recommendation**
   - Recommends products from the user's preferred category.
   - Adds personalisation through historical category preference.
   - Improves product diversity compared with the popularity-based baseline.
   - Still mainly relies on product popularity within the preferred category.

3. **Intent-Aware Recommendation**
   - Combines product performance, seller quality, high-intent lead rate, model-based lead score, category match, and user segment.
   - Better aligned with lead generation and downstream conversion objectives.
   - Designed to support recommendation strategy improvement in a live-commerce context.
   - Balances recommendation quality, lead intent, and seller reliability.

Key outputs generated:

- `fact_recommendations.csv`
- `recommendation_strategy_summary.csv`
- `recommendation_user_level.csv`
- `recommendation_category_summary.csv`

Computational note:

The recommendation logic was applied to a representative user sample to keep the notebook efficient and reproducible on a local machine. In a production environment, the same strategy logic could be scaled to the full user base using batch processing or distributed computation.

The next stage will simulate A/B testing to evaluate whether the intent-aware recommendation strategy improves CTR, CVR, inquiry rate, purchase rate, revenue per user, and GMV uplift compared with the popularity-based baseline.